<a href="https://colab.research.google.com/github/yamms2340/researchWorkCodes/blob/main/CnnModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell first to connect to your Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np

# 1. Define the blueprint so PyTorch knows how to construct the loaded weights
class Cnn1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5)
        self.c2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, dilation=2)
        self.r = nn.ReLU()
        self.fc = nn.Linear(32 * 992, 3)

    def forward(self, x):
        x = self.r(self.c1(x))
        x = self.r(self.c2(x))
        x = x.view(x.shape[0], -1)
        return self.fc(x)

# 2. Load and normalize the testing dataset
df = pd.read_csv("/content/drive/MyDrive/research_Final/lorenz_data.csv")
z_data = df['z_var'].values
z_norm = (z_data - np.min(z_data)) / (np.max(z_data) - np.min(z_data) + 1e-8)
tensor_data = torch.tensor(z_norm, dtype=torch.float32).view(1, 1, -1)

# 3. Build the brain and load the trained knowledge
model = Cnn1d()
model.load_state_dict(torch.load("/content/drive/MyDrive/research_Final/cnn_weights.pth", weights_only=True))

# 4. Predict the Exponents
model.eval()
with torch.no_grad():
    predicted_exponents = model(tensor_data).numpy()[0]

print("\n--- CNN Predicted Spectrum ---")
print(f"LE1 (Maximal):     {predicted_exponents[0]:.4f}")
print(f"LE2 (Marginal):    {predicted_exponents[1]:.4f}")
print(f"LE3 (Contractive): {predicted_exponents[2]:.4f}")

# 5. Save the CNN predictions to Drive
df_cnn = pd.DataFrame({
    "LE_Type": ["LE1 (Max)", "LE2 (Marg)", "LE3 (Cont)"],
    "CNN_Prediction": predicted_exponents
})
save_path = "/content/drive/MyDrive/research_Final/cnn_results.csv"
df_cnn.to_csv(save_path, index=False)
print(f"Successfully saved predictions to: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- CNN Predicted Spectrum ---
LE1 (Maximal):     0.8162
LE2 (Marginal):    0.0263
LE3 (Contractive): -14.8552
Successfully saved predictions to: /content/drive/MyDrive/research_Final/cnn_results.csv
